In [ ]:
import torch
import pandas as pd

path_train = "../datasets/di_pizeoelectric_tensor/dielectric_tensor_train.json"
path_val   = "../datasets/di_pizeoelectric_tensor/dielectric_tensor_val.json"

def load_tensors(path):
    df = pd.read_json(path)  # 会还原成当初的 DataFrame
    # 'dielectric_tensor' 这一列里每个元素都是 3x3 的 list
    tensors = torch.tensor(df["dielectric_tensor"].tolist(), dtype=torch.float32)
    return tensors   # [N, 3, 3]

train_t = load_tensors(path_train)
val_t   = load_tensors(path_val)

mean_t = train_t.mean(dim=0)          # [3,3]
mae_baseline = (val_t - mean_t).abs().mean().item()

print("Baseline MAE (always predict mean tensor):", mae_baseline)


Baseline MAE (always predict mean tensor): 17.108381271362305


In [ ]:
import torch
import pandas as pd
from matten.dataset.structure_scalar_tensor import TensorDataModule
from matten.model_factory.task import TensorRegressionTask
from matten.model_factory.tfn_scalar_tensor import ScalarTensorModel
from matten.tensor.cartesian_tensor import CartesianTensor

import yaml
from pathlib import Path

config_path = Path("configs/materials_tensor_dielectric.yaml")
config = yaml.safe_load(open(config_path))

# datamodule，跟训练时一样
dm = TensorDataModule(
    **config["data"],
    normalize_tensor_target=True,
    compute_dataset_statistics=True,
)
dm.prepare_data()
dm.setup("validate")

model = ScalarTensorModel.load_from_checkpoint(
    "lightning_logs/version_xxx/checkpoints/best.ckpt",  # 换成你的路径
    tasks=TensorRegressionTask(
        name=config["data"]["tensor_target_name"],
        normalize_target=True,
        dataset_statistics_path="dataset_statistics.pt",
        normalizer_kwargs={"irreps": "0e + 2e"},
    ),
    backbone_hparams=config["model"],
    dataset_hparams=dm.get_to_model_info(),
    optimizer_hparams=config["optimizer"],
    lr_scheduler_hparams=config["lr_scheduler"],
).cuda().eval()

val_loader = dm.val_dataloader()
batch = next(iter(val_loader))
batch = {k: v.cuda() for k, v in batch.items()}

with torch.no_grad():
    out = model(batch)
    y_hat_irreps = out[config["data"]["tensor_target_name"]]
    y_irreps = batch[config["data"]["tensor_target_name"]]

    task = model.tasks[0]
    y_hat_norm = y_hat_irreps
    y_norm = y_irreps

    # 反标准化到原始空间
    y_hat = task.transform_pred_metric(y_hat_norm)
    y = task.transform_target_metric(y_norm)

    # 如果想看 3x3
    ct = CartesianTensor("ij=ji")
    y_hat_cart = ct.to_cartesian(y_hat)
    y_cart = ct.to_cartesian(y)

    mae = (y_hat_cart - y_cart).abs().mean().item()
    print("manual MAE:", mae)

    print("pred[0]:\n", y_hat_cart[0])
    print("true[0]:\n", y_cart[0])

    # 看预测的方差大不大
    print("pred std over samples:", y_hat_cart.std(dim=0))


/home/qygao/.conda/envs/matten/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/qygao/.conda/envs/matten/lib/python3.10/site-packages/torchmetrics/utilities/imports.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


ModuleNotFoundError: No module named 'matten.tensor'